### Traditionelle Spektrumschätzung mit dem Periodogramm

Das **Periodogramm** ist eine einfache Methode, um das Leistungsdichtespektrum
$S_{XX}(\mathrm{e}^{j\Omega})$ auf Basis von $N$ Stichproben eines stationären Prozesses $X(k)$ zu schätzen.

$$
\mathrm{Per}_N^X(\Omega) := \frac{1}{N} \left| X(e^{j\Omega}) \right|^2,
\quad \text{mit} \quad X(e^{j\Omega}) = \mathcal{F}{ X(k) }, \quad k=0,\dots,N-1
$$

1. Erstelle zwei Modellprozesse $X_1(k), X_2(k)$ mit der Länge $N=2^{12}$ durch lineares Filtern eines weißen Gaußschen Rauschprozesses $N(k)$ mit Mittelwert null und Leistung eins unter Verwendung der Filter $H_1(z)$ und $H_2(z)$:

$$
H_1(z) = 1 - 4 z^{-1} + 6 z^{-2} - 4 z^{-3} + z^{-4}, \quad
H_2(z) = \frac{1}{1 + 0.8, z^{-1}}.
$$

2. Plotte das „wahre“ Leistungsdichtespektrum $S_{X_i X_i}(\mathrm{e}^{j\Omega})$ für beide Filter.

3. Berechne die Periodogramme $\mathrm{Per}_N^{X_i}(\Omega)$ für $N = 2^6, 2^8, 2^{10}$ und $N = 2^{12}$ Stichproben für beide Prozesse.
   Warum nimmt die Varianz der Schätzung nicht ab, wenn die Anzahl der Stichproben $N$ steigt?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lfilter
from helper_functions import lper
# BEGIN SOLUTION
N = 2**12
N_vor = 500
n = np.random.randn(N + N_vor)

# Coefficient vector for filters H1 and H2, filtering with N
h1 = [1, -4, 6, -4, 1]
x1_all = np.convolve(h1, n)
x1 = x1_all[N_vor:N_vor + N]

h2 = [1, 0.8]

# Filter H2 (IIR)
x2_all = lfilter([1], h2, n)
x2 = x2_all[N_vor:N_vor + N]

# True power spectral density for the two filters
NFFT = 2**12

Sx1x1 = np.abs(np.fft.fft(h1, NFFT))**2
axis_Sx1 = [0, 1, 0, 2 * max(Sx1x1)]

Sx2x2 = 1 / (np.abs(np.fft.fft(h2, NFFT))**2)
axis_Sx2 = [0, 1, 0, 1.5 * max(Sx2x2)]

fT = np.arange(0, 1, 1/NFFT)

# Graphical representation
plt.figure()
plt.plot(fT, Sx1x1)
plt.axis(axis_Sx1)
plt.xlabel(r'$\Omega/2\pi = f \cdot T$')
plt.ylabel(r'$S_{XX}(e^{j\cdot\Omega})$')
plt.title(r'True Power Spectral Density $S_{X1X1}$ of $x_1$')

plt.figure()
plt.plot(fT, Sx2x2)
plt.axis(axis_Sx2)
plt.xlabel(r'$\Omega/2\pi = f \cdot T$')
plt.ylabel(r'$S_{XX}(e^{j\cdot\Omega})$')
plt.title(r'True Power Spectral Density $S_{X2X2}$ of $x_2$')

input('....Press RETURN for the next task:')

# Draw periodograms and true power spectral densities
for k in [2**6, 2**8, 2**10, 2**12]:
    Sxx = lper(x1[:k], NFFT)
    plt.figure()
    plt.plot(fT, Sxx)
    plt.plot(fT, Sx1x1, '--')
    plt.xlabel(r'$\Omega/2\pi = f \cdot T$')
    plt.ylabel('Per(Om)')
    plt.title(f'Periodogram of $S_{{X1X1}}$, N={k}')
    plt.axis(axis_Sx1)
    plt.show()

input('....Press RETURN to continue:')

for k in [2**6, 2**8, 2**10, 2**12]:
    Sxx = lper(x2[:k], NFFT)
    plt.figure()
    plt.plot(fT, Sxx)
    plt.plot(fT, Sx2x2, '--')
    plt.xlabel(r'$\Omega/2\pi = f \cdot T$')
    plt.ylabel('Per(Om)')
    plt.title(f'Periodogram of $S_{{X2X2}}$, N={k}')
    plt.axis(axis_Sx2)
    plt.show()
# END SOLUTION

### Lösung: Traditionelle Spektrumschätzung mit dem Periodogramm

a) Gegeben einem Eingangssignalvektor `n = [n_0, ..., n_{N-1}]` berechnet die Python-Funktion

```python
from scipy.signal import lfilter

xi = lfilter(bi, ai, n)
```

das Ausgangssignal

```
xi = [x_{0,i}, ..., x_{N-1,i}]
```

des ARMA-Modells

$$
H_i(z) = \frac{B_i(z)}{A_i(z)} = \frac{\sum_{\mu=0}^{m} b_{\mu,i} z^{-\mu}}{\sum_{\nu=0}^{n} a_{\nu,i} z^{-\nu}},
$$

wobei die MA-Koeffizienten $b_{\mu,i}$ und die AR-Koeffizienten durch die Vektoren `bi` und `ai` gegeben sind.

Für $H_1(z)$: `b1 = [1, -4, 6, -4, 1]` und `a1 = 1`
Für $H_2(z)$: `b2 = 1` und `a2 = [1, 0.8]`

---

b) Für stationäres weißes Rauschen mit Varianz $\sigma_N^2 = 1$ gilt für die wahren Leistungsdichtespektren des Ausgangssignals $x_i(k)$:

$$
S_{X_i X_i}(e^{j\Omega}) = \sigma_N^2 \cdot |H_i(e^{j\Omega})|^2 = \left| \frac{B_i(e^{j\Omega})}{A_i(e^{j\Omega})} \right|^2 = \frac{| \mathcal{F}{b_i(k)} |^2}{| \mathcal{F}{a_i(k)} |^2}.
$$

Um die wahren Leistungsdichtespektren $S_{X_1 X_1}(e^{j\Omega})$ und $S_{X_2 X_2}(e^{j\Omega})$ darzustellen, wird die diskrete Fourier-Transformation $\mathcal{F}{\cdot}$ durch eine $2^{12}$-Punkt-FFT approximiert.

<img src="../pngs/ldsv8a02_sx1x1.png" width="250"> <img src="../pngs/ldsv8a02_sx2x2.png" width="250">

---

c) Die folgenden Abbildungen zeigen die Periodogramme $\mathrm{Per}_N^{X_i}(\Omega)$ für $N=2^6=64$, $2^8=256$, $2^{10}=1028$ und $N=2^{12}=4096$ Datenpunkte jedes Stichprobenprozesses $X_1(k)$ und $X_2(k)$.

<img src="../pngs/ldsv8a02_perx1N64.png" width="250"> <img src="../pngs/ldsv8a02_perx1N256.png" width="250"> <img src="../pngs/ldsv8a02_perx1N1024.png" width="250"> <img src="../pngs/ldsv8a02_perx1N4096.png" width="250">

<img src="../pngs/ldsv8a02_perx2N64.png" width="250"> <img src="../pngs/ldsv8a02_perx2N256.png" width="250"> <img src="../pngs/ldsv8a02_perx2N1024.png" width="250"> <img src="../pngs/ldsv8a02_perx2N4096.png" width="250">

Die diskrete Fourier-Transformation (DTFT) $\mathcal{F}{\cdot}$ wird erneut durch eine $2^{12}$-Punkte-FFT approximiert. Das wahre Leistungsdichtespektrum $S_{X_i X_i}(e^{j\Omega})$ ist durch eine gestrichelte Linie dargestellt.
Es ist erkennbar, dass die Spektrumschätzung mittels Periodogramm unbefriedigende Ergebnisse liefert, bedingt durch die Varianz

$$
\mathrm{Var}{ \mathrm{Per}*N^{X_i}(\Omega) } \sim S*{X_i X_i}(e^{j\Omega})^2.
$$

Nur in den Frequenzintervallen, in denen das wahre Leistungsdichtespektrum $S_{X_i X_i}(e^{j\Omega})$ verschwindet, liefert das Periodogramm null (mit verschwindender Varianz). In Frequenzbereichen mit Spektralanteilen erlaubt das Periodogramm keine Vorhersage von $S_{X_i X_i}(e^{j\Omega})$, da die Schätzung durch zu hohe Varianz entwertet wird. Eine Erhöhung von $N$ ändert dieses Verhalten ebenfalls nicht.
Das Bias-Verhalten verbessert sich, wenn $N$ erhöht wird.